# Imperviousness Time Series Reconstruction
## Propagating Modern Accuracy Backwards in Time

---

The **Copernicus Land Monitoring Service (CLMS)** provides geographical information on land cover and its changes, land use, vegetation state, water cycle and earth surface energy variables to a broad range of users in Europe and across the world for various domains and applications. CLMS is jointly implemented by the **European Environment Agency (EEA)** and the European Commission’s Directorate-General **Joint Research Centre (JRC)**. 

The **High-Resolution Layer (HRL) Imperviousness** is part of the pan-European CLMS portfolio that currently covers the EEA38+UK countries. It consists of two products: Imperviousness and Impervious Built-Up, along with their derived and supporting layers. All layers are derived from high-resolution optical satellite image time series (Sentinel-2) via automatic image processing methods and provide dedicated information on **Impervious and Built-Up** surfaces and detected dynamics between two reference years.
Further information on the HRL IMD can be found in the: [PUM](https://land.copernicus.eu/en/technical-library/product-user-manual-high-resolution-layer-imperviousness-2024) or [ATBD](https://land.copernicus.eu/en/technical-library/algorithm-theoretical-basis-document-high-resolution-layer-imperviousness-2024).

The High-Resolution Layers, such as **Imperviousness Density (IMD)**, are updated every three years, incorporating continuously improved algorithms and refined calibration. While recent editions leverage modern high-revisit constellations, pre-Sentinel products relied on Landsat data characterized by a native 30 m resolution and sparser observations, occasionally resulting in data gaps (e.g., in the 2006 reference year) and resolution shifts. Combining modern sensor capabilities with ongoing algorithm enhancements creates an opportunity:

> *Can we use the accuracy of the most recent layer to improve our representation of historical conditions?*

The answer is **yes** — but the approach matters. 

This notebook demonstrates why the most intuitive approach—directly subtracting detected change—introduces physically impossible values in the time series. While a full, centralized reprocessing using unified algorithms and common calibration across all historical epochs would be the gold standard, end users require practical post-processing solutions today. To address this, we introduce the **Binary Mask Substitution** method as a robust alternative. This approach relies on the conservative assumption that impervious surfaces identified in current High-Resolution Layers were already present in earlier reference years, effectively eliminating legacy data gaps across the 2006–2015 harmonized series without introducing negative density artifacts.

### What you will learn

| Section | Topic |
|---------|-------|
| **Intro** | Data products, site selection, baseline visualisation |
| **Section 1** | Why backward subtraction fails and produces negative imperviousness values |
| **Section 2** | The binary mask substitution that avoids these problems |
| **Section 3** | Interactive comparison and exploration |
| **Section 4** | Limitations and recommendations |

### Key data products

| Product | Description | Values |
|---------|------------|--------|
| **IMD** | Imperviousness Density — sealed surface fraction per pixel | 0–100 % |
| **IMDC** | Imperviousness Density Change — detected change between epochs | encoded bytes |

**IMDC encoding:** Raw values are unsigned bytes (0–255) using an offset of 100:
- `100` = no change (0 %)
- `113` = +13 % increase
- `87` = −13 % decrease
- `201` = technical no-change class
- `255` = nodata

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
import ipywidgets as widgets
import warnings

from IPython.display import display
from osgeo import gdal
import folium
from folium.raster_layers import ImageOverlay

# raster I/O, resampling, and array-to-image conversion
from helpers.raster_utils import load_array, normalize_change, match_to_grid, array_to_img, bbox_to_4326

gdal.UseExceptions()
warnings.filterwarnings('ignore', category=FutureWarning)

print('Environment ready.')


In [ ]:
# ── Timeline ──────────────────────────────────────────────────────────────────
YEARS        = ['24', '21', '18', '15', '12', '09', '06']   # newest to oldest
PAIRS        = ['2124', '1821', '1518', '1215', '0912', '0609']
NON_BASELINE = YEARS[1:]

# ── Site definitions ─────────────────────────────────────────────────────────
with open('data/site_definition.yaml', 'r') as f:
    SITES = yaml.safe_load(f)

print('Available sites:', list(SITES.keys()))


---
## Data and Site Selection

### IMD — Imperviousness Density Status Layers

Seven snapshots of sealed-surface density, one per epoch:

| Year | Role in this notebook |
|------|----------------------|
| 2024 | **Baseline** — most recent, highest quality |
| 2021–2006 | Historical reference layers |

Each pixel value represents the percentage of impervious surface (0–100 %).

### IMDC — Imperviousness Density Change Layers

Six paired change layers, each encoding detected real change between two consecutive epochs.
After normalising (subtracting the offset 100), values represent change in percentage points:
positive = new sealed surface, negative = de-sealing.

### Cross-resolution note

For the 10 m study area, the IMDC layer has a **native resolution of 20 m**. This mismatch is handled as follows:

1. Derive the binary change mask at 20 m (change / no-change)
2. Resample the **mask only** to 10 m using nearest-neighbour
3. Apply the 10 m mask to the 10 m status layer

Resampling the mask (not the raw values) is important: nearest-neighbour of a binary array yields exact copies of the original 0/1 values with no fractional artefacts.

In [ ]:
all_sites = list(SITES.keys())
region_selector = widgets.Dropdown(
    options=all_sites,
    value=all_sites[0],
    description='Region:',
    style={'description_width': 'initial'},
)

# Keep SELECTED_SITE in sync with the widget so later cells can just read the
# global instead of the dropdown; re-run the cells below after changing it.
def _on_region_change(change):
    global SELECTED_SITE
    SELECTED_SITE = change['new']

region_selector.observe(_on_region_change, names='value')
SELECTED_SITE = region_selector.value
display(region_selector)


In [ ]:
cfg        = SITES[SELECTED_SITE]
STATUS_RES = cfg['status_res']
CHANGE_RES = cfg['change_res']
XMIN, XMAX = cfg['xmin'], cfg['xmax']
YMIN, YMAX = cfg['ymin'], cfg['ymax']
ZOOM       = cfg['zoom']

CLAT, CLON, BOUNDS_4326 = bbox_to_4326(XMIN, YMIN, XMAX, YMAX, cfg['target_srid'])

# True when the change (IMDC) layer has a coarser resolution than the status
# (IMD) layer; triggers the mask-resampling step used throughout the notebook.
CROSS_RES = STATUS_RES != CHANGE_RES

# File paths for the selected site's status (IMD) and change (IMDC) rasters
def imd_path(year):  return f'data/IMD_{year}_{SELECTED_SITE}_{STATUS_RES}m.tif'
def imdc_path(pair): return f'data/IMDC_{pair}_{SELECTED_SITE}_{CHANGE_RES}m.tif'

print(f'Study area   : {SELECTED_SITE}')
print(f'Status res   : {STATUS_RES} m')
print(f'Change res   : {CHANGE_RES} m')
print(f'Extent       : {XMIN}–{XMAX} E,  {YMIN}–{YMAX} N  (EPSG:{cfg["target_srid"]})')
print(f'Center       : {CLAT:.5f} N, {CLON:.5f} E  (EPSG:4326)')


In [ ]:
# Interactive map showing the study area location in Europe
m = folium.Map(location=[CLAT, CLON], zoom_start=ZOOM, tiles='CartoDB positron')

folium.Rectangle(
    bounds=BOUNDS_4326,
    color='#c62828', weight=2.5,
    fill=True, fill_color='#ef5350', fill_opacity=0.12,
    tooltip=f'{SELECTED_SITE} study area'
).add_to(m)

folium.Marker(
    [CLAT, CLON],
    icon=folium.Icon(color='red', icon='info-sign'),
    tooltip=SELECTED_SITE
).add_to(m)

print(BOUNDS_4326)

display(m)


In [ ]:
imd24, _ = load_array(imd_path('24'))
imd24[imd24 == 255] = np.nan  # 255 is the nodata value for IMD rasters

cmap_imd = plt.get_cmap('YlOrRd').copy()
cmap_imd.set_bad('#cccccc')  # grey for nodata pixels

aspect = imd24.shape[0] / imd24.shape[1]
fig, ax = plt.subplots(figsize=(10, max(2.5, 10 * aspect)))
im = ax.imshow(np.ma.masked_invalid(imd24), cmap=cmap_imd, vmin=0, vmax=100)
plt.colorbar(im, ax=ax, shrink=0.7, label='Imperviousness (%)')
ax.set_title(f'IMD 2024 — {SELECTED_SITE}  (Baseline, {STATUS_RES} m resolution)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()


---
## Section 1: Backward Subtraction Method

### The intuitive approach

The most natural way to reconstruct the 2021 layer from the 2024 baseline is:

$$\text{IMD}_r(2021) = \text{IMD}_r(2024) - \text{IMDC}(2021 \to 2024)$$

and continue backwards for each epoch. This is straightforward to implement and appears logically sound. However, it contains a critical flaw.

### Why it fails: three overlapping problems

**1. Time-bound validity of change estimates**
Each IMDC layer is valid only between its two specific observation years. The absolute magnitude of change between 2018 and 2021 cannot be safely applied to a 2024 baseline that was produced with a completely different algorithm version.

**2. Technical corrections masquerade as real changes**
When a newer algorithm corrects a systematic error from a previous run (e.g. an overestimated 25 % pixel is corrected to 0 %), the change layer records a −25 % change. Subtracting this backwards would produce +25 % at the historical position — but that 25 % never actually existed physically.

**3. Cumulation of errors**
With each backward step, errors accumulate. A pixel at 0 % in 2024 that has +13 % change between 2018 and 2021 will be reconstructed as −13 % for 2018. **Negative imperviousness is physically impossible.**

### A worked example

Consider a single pixel tracked through the time series:

**Table 1 — Status values**

| Year | IMD (reported) | IMD reconstructed (subtraction) |
|------|---------------|--------------------------------|
| 2024 | 0 % | 0 % (= baseline) |
| 2021 | 25 % | **0 %** (should be 25 %) |
| 2018 | 0 % | **−13 %** (impossible!) |

**Table 2 — Detected changes**

| Period | Real change (IMDC) | Technical change |
|--------|-------------------|------------------|
| 2021→2024 | 0 % | +25 % (algorithm correction) |
| 2018→2021 | +13 % | +12 % |

Step-by-step reconstruction:
- `IMD_r(2024) = 0 %`
- `IMD_r(2021) = 0 − 0 = 0 %`  ← wrong (real value is 25 %)
- `IMD_r(2018) = 0 − 13 = −13 %`  ← **impossible**

The code below applies this method and visualises where values fall outside the valid 0–100 % range.

In [ ]:
print(f'Loading data for {SELECTED_SITE} ...')

# Status arrays: year -> (float array, GDAL dataset)
STATUS = {}
for year in YEARS:
    arr, ds = load_array(imd_path(year))
    arr[arr == 255] = np.nan
    STATUS[year] = (arr, ds)
print(f'  {len(STATUS)} status layers loaded')

# Resample any status layer that doesn't match the 2024 baseline grid.
# Epochs 2006–2015 are only available at 20 m natively, so they may land
# at half the expected resolution even when named _10m.tif.
_ref_shape = STATUS['24'][0].shape
_ref_ds    = STATUS['24'][1]
for _yr in YEARS[1:]:
    _arr, _ds = STATUS[_yr]
    if _arr.shape != _ref_shape:
        print(f'  Resampling IMD 20{_yr} from {_arr.shape} → {_ref_shape} (bilinear)')
        STATUS[_yr] = (match_to_grid(_arr, _ds, _ref_ds, alg=gdal.GRA_Bilinear), _ref_ds)

# Change arrays: pair -> (raw byte array, GDAL dataset)
CHANGE = {}
for pair in PAIRS:
    arr, ds = load_array(imdc_path(pair))
    CHANGE[pair] = (arr, ds)
print(f'  {len(CHANGE)} change layers loaded')

REF_DS = STATUS['24'][1]  # reference grid for resampling

s_shape = STATUS['24'][0].shape
c_shape = CHANGE['2124'][0].shape
print(f'\nStatus grid:  {s_shape[0]} x {s_shape[1]} px  ({STATUS_RES} m)')
print(f'Change grid:  {c_shape[0]} x {c_shape[1]} px  ({CHANGE_RES} m)')
if CROSS_RES:
    print('  -> Cross-resolution: change mask will be upsampled to match the status grid')


In [ ]:
print('Applying backward subtraction ...')

sub_results  = {}
sub_baseline = STATUS['24'][0].copy()

for pair, prev_year, curr_year in zip(PAIRS, YEARS[1:], YEARS[:-1]):
    raw_change, ch_ds = CHANGE[pair]
    norm_change = normalize_change(raw_change)

    # Resample normalised change values to status grid when resolutions differ
    if CROSS_RES:
        norm_change = match_to_grid(norm_change, ch_ds, REF_DS)

    result = sub_baseline - norm_change
    result[np.isnan(STATUS[prev_year][0])] = np.nan  # preserve nodata

    sub_results[prev_year] = result
    sub_baseline = result.copy()

# Tally invalid pixels per year
INVALID = {}
for y in NON_BASELINE:
    arr     = sub_results[y]
    n_inv   = int(np.nansum((arr < 0) | (arr > 100)))
    n_total = int(np.sum(~np.isnan(arr)))
    pct     = 100 * n_inv / n_total if n_total else 0
    INVALID[y] = (n_inv, n_total, pct)
    print(f'  20{y}: {n_inv:6,} invalid pixels  ({pct:.2f} % of total)')

In [ ]:
cmap_imd = plt.get_cmap('YlOrRd').copy()
cmap_imd.set_bad('#cccccc')

legend_els = [
    mpatches.Patch(color=(0.18, 0.20, 0.88), label='Impossible negative  (< 0 %)'),
    mpatches.Patch(color=(0.05, 0.65, 0.20), label='Above maximum  (> 100 %)'),
]

for year in NON_BASELINE:
    arr    = sub_results[year]
    aspect = arr.shape[0] / arr.shape[1]
    fig, ax = plt.subplots(figsize=(14, max(2.5, 14 * aspect)))

    im = ax.imshow(np.ma.masked_invalid(np.clip(arr, 0, 100)),
                   cmap=cmap_imd, vmin=0, vmax=100)
    plt.colorbar(im, ax=ax, shrink=0.7, label='%')

    # Overlay the out-of-range pixels in solid colour on top of the clipped map
    rgba = np.zeros((*arr.shape, 4), dtype=float)
    rgba[(~np.isnan(arr)) & (arr < 0)]   = [0.18, 0.20, 0.88, 0.82]  # blue
    rgba[(~np.isnan(arr)) & (arr > 100)] = [0.05, 0.65, 0.20, 0.82]  # green (more visible against YlOrRd than purple)
    ax.imshow(rgba)

    n_inv, _, pct = INVALID[year]
    ax.set_title(
        f'Backward Subtraction — 20{year}   ({n_inv:,} invalid px, {pct:.1f} %)\n'
        'blue = negative imperviousness   |   green = above 100 %',
        fontsize=12
    )
    ax.axis('off')
    ax.legend(handles=legend_els, loc='lower right', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    plt.show()


In [ ]:
# Shared y-axis range (based on interior 1-99 % counts) so all years are directly comparable
_shared_max = 0
for year in NON_BASELINE:
    flat     = sub_results[year].flatten()
    interior = flat[(flat > 0) & (flat < 100)]
    if len(interior):
        counts, _ = np.histogram(interior, bins=48, range=(1, 99))
        _shared_max = max(_shared_max, counts.max())

for year in NON_BASELINE:
    flat = sub_results[year].flatten()
    flat = flat[~np.isnan(flat)]

    valid = flat[(flat >= 0) & (flat <= 100)]
    neg   = flat[flat < 0]
    over  = flat[flat > 100]

    fig, ax = plt.subplots(figsize=(14, 4))

    ax.hist(valid, bins=50, range=(0, 100), color='#777', alpha=0.7, label='Valid (0–100 %)')
    if len(neg):
        ax.hist(neg,  bins=20, color='#2f3fd4', alpha=0.85, label=f'Negative: {len(neg):,} px')
    if len(over):
        ax.hist(over, bins=20, color='#c41bc4', alpha=0.85, label=f'>100%%: {len(over):,} px')

    ax.set_ylim(0, _shared_max * 1.2)

    ax.axvline(0,   color='black', lw=1.5, ls='--', alpha=0.8)
    ax.axvline(100, color='black', lw=1.5, ls=':',  alpha=0.8)
    ax.set_title(
        f'Pixel Value Distribution — Backward Subtraction 20{year}',
        fontsize=12
    )
    ax.set_xlabel('IMD value (%)')
    ax.set_ylabel('Pixel count')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


### Discussion

The histograms and maps above show a clear trend: **the further back in time, the more invalid pixels accumulate**. This is the mathematical consequence of the cumulation issue — every backward step can push additional pixels outside the valid 0–100 % range.

The fundamental problem is that simple subtraction **treats the IMDC layer as an absolute quantity** — which it is not. The change layer tells us that something changed, and by how much *in relative terms between those two specific years*. It does not tell us what the "true" historical value should be, especially when algorithm updates have changed the baseline.

The next section presents a method that avoids this entirely.

---
## Section 2: Binary Mask Substitution

### The core idea: use change as a switch, not a value

Instead of asking *how much* did the surface change, we ask *did it change at all?*

For each pixel and each backward step:

| IMDC says... | Physical interpretation | Action |
|--------------|------------------------|--------|
| **Real change** (IMDC ≠ 0) | The surface physically changed between these two years | Adopt the **original historical status** value |
| **No real change** (IMDC = 0) | The surface stayed the same | Keep the **modern baseline** value (it is more accurate) |

By never subtracting a quantity, the output is always a copy of a real, valid status value. **Negative values are impossible by construction.**

### Handling the cross-resolution case (Innsbruck: 20 m IMDC + 10 m IMD)

When the change layer is coarser than the status layer:

1. Compute the binary mask (change / no-change) at the **native 20 m** IMDC resolution
2. Resample the **mask** to **10 m** using nearest-neighbour
3. Apply to the 10 m status and baseline layers

Each 10 m pixel simply inherits the change flag of its enclosing 20 m cell. No interpolation of change values occurs.

### Pseudocode

```python
running_baseline = IMD_2024  # start with the most accurate layer

for step in [2021, 2018, 2015, 2012, 2009, 2006]:
    # 1. Derive binary mask at native IMDC resolution
    norm_change = normalize(IMDC[step, next_step])
    change_mask = (norm_change != 0)          # True where real change occurred

    # 2. Resample mask to status grid (nearest-neighbour)
    change_mask = resample_NN(change_mask, target_resolution)

    # 3. Binary mask substitution
    result = running_baseline.copy()
    result[change_mask] = IMD_historical[step][change_mask]

    running_baseline = result  # carry forward for next step
```

Note: the exact IMDC value is **never used**. Only the presence or absence of change matters. Therefore, the masking can also be done using the classified change layer (IMCC).

In [ ]:
print('Applying binary mask substitution ...')

ind_results  = {}
ind_baseline = STATUS['24'][0].copy()

for pair, prev_year, curr_year in zip(PAIRS, YEARS[1:], YEARS[:-1]):
    raw_change, ch_ds = CHANGE[pair]
    hist_status       = STATUS[prev_year][0]

    # Build binary change mask at native IMDC resolution
    norm_change = normalize_change(raw_change)
    mask_native = (norm_change != 0).astype(np.float32)  # float needed for GDAL write

    # Resample mask to status grid (NN preserves binary values exactly)
    if CROSS_RES:
        mask_resampled = match_to_grid(mask_native, ch_ds, REF_DS) > 0.5
    else:
        mask_resampled = mask_native.astype(bool)

    # binary mask substitution: where change occurred, adopt historical value
    result       = ind_baseline.copy()
    valid_change = mask_resampled & ~np.isnan(hist_status)
    result[valid_change] = hist_status[valid_change]

    ind_results[prev_year] = result
    ind_baseline = result.copy()
    print(f'  20{prev_year}: {int(np.sum(valid_change)):,} pixels updated from historical layer')

# Verify: binary mask substitution should produce zero invalid pixels
all_ok = True
for y in NON_BASELINE:
    arr   = ind_results[y]
    n_inv = int(np.nansum((arr < 0) | (arr > 100)))
    if n_inv:
        print(f'  WARNING 20{y}: {n_inv} unexpected invalid pixels')
        all_ok = False
if all_ok:
    print('Verification: all binary mask substitution results are within valid 0-100 % range.')

In [ ]:
VIS_YEAR = '06'  # choose from: '21' | '18' | '15' | '12' | '09' | '06'

assert VIS_YEAR in NON_BASELINE, f"Invalid year. Choose from: {NON_BASELINE}"

orig          = STATUS[VIS_YEAR][0]
sub           = sub_results[VIS_YEAR]
ind           = ind_results[VIS_YEAR]
n_inv, _, pct = INVALID[VIS_YEAR]

cmap_imd = plt.get_cmap('YlOrRd').copy()
cmap_imd.set_bad('#cccccc')

aspect = orig.shape[0] / orig.shape[1]
row_h  = max(2.5, 14 * aspect)
fig, axes = plt.subplots(4, 1, figsize=(14, row_h * 4))

for ax, arr, title in zip(
    axes[:3],
    [orig, sub, ind],
    [f'Original IMD 20{VIS_YEAR}',
     f'Subtraction 20{VIS_YEAR}  ({n_inv:,} invalid px, {pct:.1f} %)',
     f'Binary Mask Substitution 20{VIS_YEAR}'],
):
    im = ax.imshow(np.ma.masked_invalid(np.clip(arr, 0, 100)),
                   cmap=cmap_imd, vmin=0, vmax=100)
    plt.colorbar(im, ax=ax, shrink=0.7, label='%')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

    # Only the subtraction panel can contain out-of-range values; highlight them
    if 'Subtraction' in title:
        rgba = np.zeros((*arr.shape, 4), dtype=float)
        rgba[(~np.isnan(arr)) & (arr < 0)]   = [0.18, 0.20, 0.88, 0.78]
        rgba[(~np.isnan(arr)) & (arr > 100)] = [0.05, 0.65, 0.20, 0.78]
        ax.imshow(rgba)

# Difference panel: where do the two reconstruction methods disagree? (comment 12)
diff = ind - sub
dmax = np.nanmax(np.abs(diff)) if np.any(~np.isnan(diff)) else 1
im_d = axes[3].imshow(diff, cmap='RdBu_r', vmin=-dmax, vmax=dmax)
plt.colorbar(im_d, ax=axes[3], shrink=0.7, label='pp difference')
axes[3].set_title(f'Difference — Binary Mask Substitution minus Subtraction 20{VIS_YEAR}', fontsize=12)
axes[3].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar chart: invalid pixel counts per year ──────────────────────────────────
inv_counts = [INVALID[y][0] for y in NON_BASELINE]
pct_labels = [f'{INVALID[y][2]:.1f}%' for y in NON_BASELINE]

bars = ax1.bar(range(len(NON_BASELINE)), inv_counts,
               color='#2f3fd4', alpha=0.50, label='Subtraction method')
ax1.bar(range(len(NON_BASELINE)), [0] * len(NON_BASELINE),
        color='#27ae60', alpha=0.50, label='Binary mask substitution (always 0)')
ax1.bar_label(bars, labels=pct_labels, padding=4, fontsize=9)
ax1.set_xticks(range(len(NON_BASELINE)))
ax1.set_xticklabels([f'20{y}' for y in NON_BASELINE])
ax1.set_ylabel('Number of invalid pixels')
ax1.set_title('Invalid Pixel Count per Reconstructed Year', fontsize=12)
ax1.legend(fontsize=10)
ax1.set_ylim(0, max(max(inv_counts) * 1.20, 1))

fig.text(
    0.5, -0.03,
    f'Pixel counts refer to the {SELECTED_SITE} study-area tile only (the extent shown in the maps above), '
    'not the full product coverage.',
    ha='center', fontsize=9, style='italic'
)

# ── Distribution comparison for the most affected year ────────────────────────
worst = max(NON_BASELINE, key=lambda y: INVALID[y][0])
bins  = np.linspace(-30, 110, 70)

for arr, label, color in [
    (STATUS[worst][0],    f'Original IMD 20{worst}', '#888'),
    (sub_results[worst],  'Subtraction',              '#2f3fd4'),
    (ind_results[worst],  'Binary mask substitution', '#27ae60'),
]:
    flat = arr.flatten()
    flat = flat[~np.isnan(flat)]
    ax2.hist(flat, bins=bins, alpha=0.55, color=color, label=label)

ax2.axvline(0,   color='black', lw=1.5, ls='--', alpha=0.8)
ax2.axvline(100, color='black', lw=1.5, ls=':',  alpha=0.8)
ax2.set_xlabel('Imperviousness value (%)')
ax2.set_ylabel('Pixel count')
ax2.set_title(f'Pixel Distributions — 20{worst} (most affected year)', fontsize=12)

# Scale the y-axis to the interior (1-99 %) distribution of the original layer
# so the large 0 %/100 % spikes don't dominate the plot
interior = STATUS[worst][0]
interior = interior[~np.isnan(interior)]
interior = interior[(interior > 0) & (interior < 100)]
ref_counts, _ = np.histogram(interior, bins=48, range=(1, 99))
ax2.set_ylim(0, ref_counts.max() * 1.2)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## Section 3: Interactive Exploration

Use the configuration cell below to select a year of interest. The following cells will:

1. Show a static side-by-side map comparison for that year
2. Create an interactive folium map where you can toggle between all three layers
3. Allow pixel-level value inspection by clicking on the map

Re-run the cells after changing `SELECTED_YEAR`.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  YEAR SELECTION  – modify this cell and re-run below   ║
# ║  Options: '21' | '18' | '15' | '12' | '09' | '06'     ║
# ╚══════════════════════════════════════════════════════════╝

SELECTED_YEAR = '06'

In [ ]:
year   = SELECTED_YEAR
orig_y = STATUS[year][0]
sub_y  = sub_results[year]
ind_y  = ind_results[year]

# Interactive layer map — toggle between Original, Subtraction, and Binary Mask Substitution
m = folium.Map(location=[CLAT, CLON], zoom_start=ZOOM + 1)

ImageOverlay(
    image=array_to_img(orig_y, 'YlOrRd', 0, 100),
    bounds=BOUNDS_4326, opacity=1,
    name=f'Original IMD 20{year}'
).add_to(m)

ImageOverlay(
    image=array_to_img(sub_y, 'YlOrRd', 0, 100, clr_below_vmin='#2f3fd4', clr_above_vmax='#c41bc4'),
    bounds=BOUNDS_4326, opacity=1,
    name=f'Subtraction 20{year}'
).add_to(m)

ImageOverlay(
    image=array_to_img(ind_y, 'YlOrRd', 0, 100),
    bounds=BOUNDS_4326, opacity=1,
    name=f'Binary Mask Substitution 20{year}'
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
display(m)


---
## Section 4: Limitations

The binary mask substitution is effective for improving the visual consistency of a time series. Before using it, however, users must understand its key limitations.

---

### 1. Change detection errors propagate backwards

The reconstruction inherits any errors present in the IMDC layers:

- **Change omissions**: If a real change occurred but was not detected, the modern baseline is propagated backward. A building constructed in 2009 that was missed by the change detector will appear in the 2006 reconstruction as a "ghost" impervious surface — even though it did not physically exist then.

- **Change commissions**: If a spurious change is detected where none occurred, the reconstruction incorrectly adopts the historical status value, potentially introducing visual artefacts.

- **Baseline errors**: Any error already present in IMD 2024 is carried unchanged into every earlier year where no real change was detected.

The quality of the reconstruction is bounded by the quality of the change detection.

---

### 2. This method is for visual quality — not for change quantification

The reconstructed layers are designed to provide a **spatially consistent and visually coherent** time series. They are explicitly **not** intended for quantitative change analysis.

> **If you need to measure how much imperviousness has changed, always use the original IMDC layers directly.**

| Question | Correct tool |
|----------|--------------
| How does the map look for 2009? | Binary mask substitution IMD layer |
| How many hectares of new impervious surface appeared 2009–2012? | IMDC layer (aggregate the continuous density change values) |
| What is the total sealed surface area in 2015? | Original IMD 2015 layer |

Do **not** count pixels in a classified change product. Do **not** subtract status layers from each other. Always aggregate the continuous density change values from the IMDC layer.

---

### 3. Uncertainty increases with temporal distance from the baseline

The 2021 reconstruction carries uncertainty from one change epoch. The 2006 reconstruction carries accumulated uncertainty from all six. Treat the earliest reconstructed layers with more caution.

---

### 4. Spatial precision of changed areas is limited by the change layer resolution

For sites with a coarser IMDC layer (e.g. 20 m IMDC used with 10 m IMD status), the **boundaries** of changed areas in the reconstruction are limited to 20 m precision. The 10 m IMD values within those areas are correct, but the spatial sharpness of change edges is determined by the 20 m mask.

---

### 5. Why not a full reprocessing?

A full reprocessing with the current algorithm and common calibration across all epochs would in principle be the most correct solution. Binary mask substitution is a lower-cost alternative to that; it assumes a pixel's current status is representative of its historical state wherever no real change was detected. Its main benefit is on the already harmonised 2006–2015 layers, where it removes data gaps and increases the resolution to 10m from that period without a new processing run.

---

### Summary

| Use case | Use binary mask substitution? |
|----------|------------------------------|
| Visualise historical imperviousness maps | **Yes** |
| Create a consistent time series for cartographic products | **Yes** |
| Quantify annual imperviousness change | **No** — use IMDC (aggregate density change) |
| Compute total impervious area per year | **No** — use original IMD |
| Detect individual land-cover change events | **No** — use IMDC |
| Pixel-by-pixel "historical change history" | With caution — errors propagate |